# Performance evals for the datasheet agent (TruLens)

Evaluates `answer_question()` from `agentic_rag.ipynb` on a golden set of datasheet questions.

| Metric | Type | What it measures |
|---|---|---|
| **Correctness** (ground-truth agreement) | LLM judge vs expected answer | Is the answer factually right? |
| **Groundedness** | LLM judge, answer vs retrieved contexts | Is every claim supported by what the agent read? (hallucination check) |
| **Answer relevance** | LLM judge, question vs answer | Does it answer the question that was asked? |
| **Context relevance** | LLM judge, question vs each retrieved chunk (mean) | How focused is the agent's retrieval? |
| **Page citation recall** | deterministic | Fraction of expected source pages cited as `[p.N]` |
| Latency, tokens, tool calls, verifier verdict, revisions | from the agent run | Cost and behaviour |

The first four are TruLens's *RAG triad* plus ground truth. Because the agent decides its own retrieval, the "retrieved contexts" are everything its tools returned during the run (the evidence ledger).

**Prerequisites:** `docker compose up -d` (Qdrant + Phoenix), the collection ingested, `uv sync`, and the `.venv` kernel selected. A full run of the 13 questions takes about 10–15 minutes (agent ≈ 30–60 s per question, then the judges).

## 1. Load the agent

In [1]:
RUN_DEMO = False                 # skip the demo questions inside the agent notebook
%run -i agentic_rag.ipynb

Tracing to http://localhost:6006  project='datasheet-agent'


Datasheets: ['5V, 10A Synchronous Step-Down Silent Switcher 2 in 3mm × 3mm LQFN']
Tools: ['list_documents', 'search_datasheets', 'get_table', 'get_page', 'get_neighbor_chunks', 'view_page_image', 'calculate']


## 2. Golden dataset

Expected answers and pages were taken from the datasheet. The questions cover the behaviours that matter for this agent:
- spec lookup (guaranteed vs typical);
- device-variant awareness;
- ordering information;
- absolute maximum ratings;
- a pin function combined with a spec;
- reading a graph;
- a multi-step design calculation;
- variant comparison;
- questions the datasheet *cannot* answer, where the agent must say so rather than guess.

In [2]:
GOLDEN = [
    {"category": "spec lookup", "query": "What is the regulated feedback voltage (VFB) and its guaranteed tolerance?",
     "expected_response": "VFB is 500 mV typical, with guaranteed limits of 495 mV minimum and 505 mV maximum over the full operating temperature range.",
     "expected_pages": [3]},
    {"category": "spec lookup", "query": "What are the VIN undervoltage lockout threshold limits and the hysteresis?",
     "expected_response": "VIN UVLO (VIN rising) is 2.0 V min, 2.1 V typical, 2.2 V max over temperature; UVLO hysteresis is 150 mV typical.",
     "expected_pages": [3]},
    {"category": "variant", "query": "What output voltage does the LTC3310S-1 regulate to, and how accurately?",
     "expected_response": "The LTC3310S-1 (fixed-output version) regulates to 1.000 V typical, with 0.990 V minimum and 1.010 V maximum at IOUT = 4 A over the full temperature range.",
     "expected_pages": [3]},
    {"category": "ordering", "query": "What is the part marking for LTC3310HV-1#PBF and what is its temperature range?",
     "expected_response": "Part marking LHMP; temperature range –40°C to 150°C; 18-lead 3mm × 3mm LQFN package.",
     "expected_pages": [3]},
    {"category": "absolute max", "query": "What is the absolute maximum rating for VIN, and what is the maximum reflow temperature?",
     "expected_response": "VIN absolute maximum is –0.3 V to 6 V; maximum reflow (package body) temperature is 260°C. Absolute maximum ratings are stress ratings, not operating conditions.",
     "expected_pages": [2]},
    {"category": "spec lookup", "query": "What is the guaranteed top switch current limit?",
     "expected_response": "Top switch current limit is 14.5 A minimum, 16 A typical, 18.5 A maximum over temperature, specified for VOUT/VIN ≤ 0.2 with current out of SW.",
     "expected_pages": [4]},
    {"category": "spec lookup", "query": "What switching frequency does the part run at if RT is tied to VIN?",
     "expected_response": "With RT = VIN the default frequency is 2 MHz typical (1.8 MHz min, 2.2 MHz max over temperature).",
     "expected_pages": [4]},
    {"category": "pin + spec", "query": "What does the PGOOD pin do, and at what output level does it go high?",
     "expected_response": "PGOOD is an open-drain power-good output of an internal comparator; it is pulled low when the output is out of regulation or the part is in shutdown (with VIN above 2.25 V). The PGOOD rising threshold is 98% of the regulated VOUT typical (97% min, 99% max), with 1% hysteresis.",
     "expected_pages": [4, 10]},
    {"category": "design calculation", "query": "For the LTC3310S at 1 MHz switching frequency, 5 V input and 1.2 V output: what RT resistor should be used and what minimum inductor value does the datasheet guideline give?",
     "expected_response": "RT = 549 kΩ for 1 MHz (Table 1; RT = 568·fSW^-1.08). Since VOUT/VIN(MAX) = 0.24 ≤ 0.5, L ≥ VOUT/(3A·fSW)·(1 − VOUT/VIN(MAX)) = 1.2/(3·1)·(1 − 1.2/5) ≈ 0.304 µH.",
     "expected_pages": [14, 15]},
    {"category": "graph reading", "query": "From the typical performance curves, approximately what is the PMOS switch on-resistance at 5 V input?",
     "expected_response": "Approximately 12 mΩ (about 12–13 mΩ) at VIN = 5 V from the Switch On-Resistance vs input voltage graph; this is a typical, approximate graph value.",
     "expected_pages": [6]},
    {"category": "comparison", "query": "What is the difference between the LTC3310S, LTC3310 and LTC3310S-1?",
     "expected_response": "LTC3310S uses Silent Switcher 2 (SS2) with adjustable output; LTC3310 uses Silent Switcher 1 (SS1) with adjustable output; the -1 versions (LTC3310S-1, LTC3310-1) have a fixed 1 V output.",
     "expected_pages": [1]},
    {"category": "not in datasheet", "query": "What is the unit price of the LTC3310S in 1k quantities?",
     "expected_response": "The datasheet does not contain pricing information.",
     "expected_pages": []},
    {"category": "not in datasheet", "query": "What is the maximum input voltage of the LTC7151S?",
     "expected_response": "The LTC7151S is not covered by the datasheets in the knowledge base, so this cannot be answered from them.",
     "expected_pages": []},
]
GOLDEN_BY_QUERY = {g["query"]: g for g in GOLDEN}
print(len(GOLDEN), "questions:", dict(__import__("collections").Counter(g["category"] for g in GOLDEN)))

13 questions: {'spec lookup': 4, 'variant': 1, 'ordering': 1, 'absolute max': 1, 'pin + spec': 1, 'design calculation': 1, 'graph reading': 1, 'comparison': 1, 'not in datasheet': 2}


## 3. Judge model: a TruLens provider on the OpenAI Responses API

TruLens's own OpenAI provider package needs `openai<2`, which conflicts with Agent Framework (`openai` 3.x). So the judge (`datasheet_rag/trulens_judge.py`) plugs into TruLens through its `LLMProvider` base class, and all of TruLens's built-in feedback prompts work unchanged. The judge model is separate from the agent model, and the GPT-5 family takes `reasoning_effort` rather than `temperature`.

In [3]:
import re, time, warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=DeprecationWarning)
from trulens.core import TruSession, Feedback
from trulens.core.database.connector.default import DefaultDBConnector
from trulens.core.feedback.selector import Selector
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes
from trulens.feedback import GroundTruthAgreement
from trulens.apps.app import TruApp
from datasheet_rag.trulens_judge import ResponsesJudge

JUDGE_MODEL  = "gpt-5.4-mini"
JUDGE_EFFORT = "low"

judge = ResponsesJudge(model_engine=JUDGE_MODEL, reasoning_effort=JUDGE_EFFORT)
print("judge smoke test:", judge.relevance_with_cot_reasons("What is VFB?", "VFB is 500 mV typical.")[0])

Package jsonschema not present in requirements.


judge smoke test: 0.6666666666666666


## 4. TruLens session and instrumented app

`DatasheetQAApp.query()` is the recorded entry point. After the agent finishes, `_retrieved_contexts()` publishes the agent's evidence as a RETRIEVAL span, which is what the context-relevance and groundedness feedbacks read. Agent statistics are kept in `RUN_LOG` for the per-question report.

**Tracing note:** OpenTelemetry allows a single global tracer provider. Phoenix was registered first (by the agent notebook), so TruLens attaches its span processor to that same provider. Agent traces keep going to Phoenix (project `datasheet-agent`), and TruLens records its spans into its own database.

In [4]:
EVAL_DIR = Path("evals"); EVAL_DIR.mkdir(exist_ok=True)
session = TruSession(connector=DefaultDBConnector(database_url=f"sqlite:///{(EVAL_DIR / 'trulens.sqlite').as_posix()}"))
RESET_DB = False            # True = wipe previous eval runs
if RESET_DB:
    session.reset_database()

MAX_CONTEXTS = 15           # cap per question for the per-context relevance judge (cost control)
RUN_LOG: list[dict] = []

class DatasheetQAApp:
    @instrument(span_type=SpanAttributes.SpanType.RETRIEVAL,
                attributes={SpanAttributes.RETRIEVAL.QUERY_TEXT: "question",
                            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return"})
    def _retrieved_contexts(self, question: str, contexts: list[str]) -> list[str]:
        return contexts[:MAX_CONTEXTS]

    @instrument()
    def query(self, question: str) -> str:
        result = answer_question(question)
        self._retrieved_contexts(question, result.contexts)
        RUN_LOG.append({
            "query": question, "app_version": CURRENT_VERSION, "latency_s": result.latency_s,
            "tokens": result.usage.get("total_token_count"), "tool_calls": len(result.tool_calls),
            "tools": " → ".join(t["tool"] for t in result.tool_calls),
            "verdict": result.verdict.verdict if result.verdict else "not_verified", "revisions": result.revisions,
            "found_in_datasheet": result.answer.found_in_datasheet, "confidence": result.answer.confidence,
            "n_contexts": len(result.contexts), "phoenix_trace_id": result.trace_id,
        })
        return result.answer.answer

Overriding of current TracerProvider is not allowed


## 5. Feedback functions

In [5]:
_CITE = re.compile(r"\[p\.?\s*(\d+)(?:\s*[-–]\s*(\d+))?\]", re.I)

def page_citation_recall(question: str, answer: str) -> float:
    """Deterministic: share of the golden 'expected_pages' that the answer cites as [p.N] (or [p.N–M])."""
    expected = set(GOLDEN_BY_QUERY.get(question, {}).get("expected_pages", []))
    if not expected:
        return 1.0                                    # nothing to cite (e.g. not-in-datasheet questions)
    cited = set()
    for a, b in _CITE.findall(answer):
        cited.update(range(int(a), int(b or a) + 1))
    return len(expected & cited) / len(expected)

ground_truth = GroundTruthAgreement([{"query": g["query"], "expected_response": g["expected_response"]} for g in GOLDEN],
                                    provider=judge)

FEEDBACKS = [
    Feedback(ground_truth.agreement_measure, name="Correctness").on_input_output(),
    # answerability-aware: a correct "the datasheet doesn't say" is grounded, not a hallucination
    Feedback(judge.groundedness_measure_with_cot_reasons_consider_answerability, name="Groundedness")
        .on({"source": Selector.select_context(collect_list=True), "statement": Selector.select_record_output(),
             "question": Selector.select_record_input()}),
    Feedback(judge.relevance_with_cot_reasons, name="Answer relevance").on_input_output(),
    Feedback(judge.context_relevance_with_cot_reasons, name="Context relevance")
        .on({"question": Selector.select_record_input(), "context": Selector.select_context(collect_list=False)})
        .aggregate(np.mean),
    Feedback(page_citation_recall, name="Page citation recall").on_input_output(),
]
print([f.name for f in FEEDBACKS])

Metric implementation <function page_citation_recall at 0x000001B15DB900F0> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


['Correctness', 'Groundedness', 'Answer relevance', 'Context relevance', 'Page citation recall']


## 6. Run the evaluation

`VARIANTS` sets which agent configurations to evaluate. Each becomes a TruLens `app_version`, so the leaderboard compares them side by side. The default evaluates the full agent. Enable the second variant to measure what the verifier contributes; it doubles run time and cost.

In [6]:
VARIANTS = [
    {"app_version": f"{AGENT_MODEL}-verifier", "settings": {"ENABLE_VERIFIER": True}},
    # {"app_version": f"{AGENT_MODEL}-no-verifier", "settings": {"ENABLE_VERIFIER": False}},
    # {"app_version": "gpt-5.4-mini-verifier", "settings": {"AGENT_MODEL": "gpt-5.4-mini", "ENABLE_VERIFIER": True}},
]

app = DatasheetQAApp()
RUN_STARTED = pd.Timestamp.now()        # results below are filtered to this run (the DB keeps earlier runs for history)
for variant in VARIANTS:
    CURRENT_VERSION = variant["app_version"]
    saved = {k: globals()[k] for k in variant["settings"]}
    globals().update(variant["settings"])               # the agent reads these module-level settings at call time
    try:
        tru_app = TruApp(app, app_name="datasheet-agent", app_version=CURRENT_VERSION,
                         feedbacks=FEEDBACKS, main_method=app.query)
        for i, g in enumerate(GOLDEN, 1):
            t0 = time.perf_counter()
            with tru_app:
                app.query(g["query"])
            print(f"[{CURRENT_VERSION}] {i}/{len(GOLDEN)} {g['category']:<18} {time.perf_counter() - t0:5.1f}s  {g['query'][:70]}")
    finally:
        globals().update(saved)
session.force_flush()

[gpt-5.5-verifier] 1/13 spec lookup         25.4s  What is the regulated feedback voltage (VFB) and its guaranteed tolera


[gpt-5.5-verifier] 2/13 spec lookup         41.0s  What are the VIN undervoltage lockout threshold limits and the hystere


[gpt-5.5-verifier] 3/13 variant             39.5s  What output voltage does the LTC3310S-1 regulate to, and how accuratel


[gpt-5.5-verifier] 4/13 ordering            29.1s  What is the part marking for LTC3310HV-1#PBF and what is its temperatu


[gpt-5.5-verifier] 5/13 absolute max        17.9s  What is the absolute maximum rating for VIN, and what is the maximum r


[gpt-5.5-verifier] 6/13 spec lookup         26.1s  What is the guaranteed top switch current limit?


... (4 outputs left out)

[gpt-5.5-verifier] 11/13 comparison          54.2s  What is the difference between the LTC3310S, LTC3310 and LTC3310S-1?


[gpt-5.5-verifier] 12/13 not in datasheet    31.3s  What is the unit price of the LTC3310S in 1k quantities?


[gpt-5.5-verifier] 13/13 not in datasheet    16.9s  What is the maximum input voltage of the LTC7151S?


True

## 7. Wait for the feedback results

In [7]:
def wait_for_feedbacks(app_versions: list[str], since: pd.Timestamp, timeout_s: int = 1200) -> pd.DataFrame:
    # Context relevance has nothing to score when the agent retrieved no contexts (e.g. a part not in the knowledge base),
    # so it is only awaited for records that have contexts.
    names = [f.name for f in FEEDBACKS]
    no_ctx = {(r["query"], r["app_version"]) for r in RUN_LOG if r["n_contexts"] == 0}
    t0 = time.time()
    while True:
        records, _ = session.get_records_and_feedback()
        records = records[records["app_version"].isin(app_versions) & (pd.to_datetime(records["ts"]) >= since)]
        expected = len(GOLDEN) * len(app_versions)
        pending = 0
        for _, r in records.iterrows():
            q = json.loads(r["input"]) if str(r["input"]).startswith('"') else r["input"]
            needed = [n for n in names if not (n == "Context relevance" and (q, r["app_version"]) in no_ctx)]
            pending += sum(1 for n in needed if n not in r or pd.isna(r[n]))
        done = len(records) >= expected and pending == 0
        if done or time.time() - t0 > timeout_s:
            print(f"{len(records)}/{expected} records, pending feedbacks: {pending}")
            return records
        time.sleep(10)

versions = [v["app_version"] for v in VARIANTS]
records = wait_for_feedbacks(versions, since=RUN_STARTED)

13/13 records, pending feedbacks: 0


## 8. Results

In [8]:
FB = ["Correctness", "Groundedness", "Answer relevance", "Context relevance", "Page citation recall"]
run_log = pd.DataFrame(RUN_LOG)
golden_df = pd.DataFrame(GOLDEN)[["query", "category"]]
results = (records[["app_version", "input", "output"] + FB].rename(columns={"input": "query"})
           .assign(query=lambda d: d["query"].map(lambda s: json.loads(s) if isinstance(s, str) and s.startswith('"') else s))
           .merge(golden_df, on="query", how="left")
           .merge(run_log, on=["query", "app_version"], how="left"))

leaderboard = results.groupby("app_version").agg(
    **{m: (m, "mean") for m in FB},
    latency_s=("latency_s", "mean"), tokens=("tokens", "mean"), tool_calls=("tool_calls", "mean"),
    verified=("verdict", lambda s: (s == "supported").mean()), revisions=("revisions", "sum"), questions=("query", "count"))
display(leaderboard.round(3).T)

app_version,gpt-5.5-verifier
Correctness,0.992
Groundedness,0.902
Answer relevance,0.846
Context relevance,0.622
Page citation recall,0.962
latency_s,32.431
tokens,37056.692
tool_calls,5.000
verified,1.000
revisions,2.000


In [9]:
by_category = results.groupby(["app_version", "category"])[["Correctness", "Groundedness", "Page citation recall"]].mean()
display(by_category.round(2))

cols = ["category", "query", "Correctness", "Groundedness", "Answer relevance", "Context relevance",
        "Page citation recall", "verdict", "revisions", "tool_calls", "latency_s", "tokens"]
display(results[cols].sort_values("Correctness").style.format(precision=2)
        .background_gradient(subset=FB, cmap="RdYlGn", vmin=0, vmax=1))

Correctness  Groundedness  \
app_version      category                                        
gpt-5.5-verifier absolute max                1.0          1.00   
                 comparison                  1.0          0.67   
                 design calculation          1.0          0.89   
                 graph reading               1.0          0.67   
                 not in datasheet            1.0          0.92   
                 ordering                    1.0          1.00   
                 pin + spec                  0.9          1.00   
                 spec lookup                 1.0          0.92   
                 variant                     1.0          1.00   

                                     Page citation recall  
app_version      category                                  
gpt-5.5-verifier absolute max                         1.0  
                 comparison                           1.0  
                 design calculation                   0.5  
                 graph reading                        1.0  
                 not in datasheet                     1.0  
                 ordering                             1.0  
                 pin + spec                           1.0  
                 spec lookup                          1.0  
                 variant                              1.0

,category,query,Correctness,Groundedness,Answer relevance,Context relevance,Page citation recall,verdict,revisions,tool_calls,latency_s,tokens
7,pin + spec,"What does the PGOOD pin do, and at what output level does it go high?",0.90,1.00,1.00,0.67,1.00,supported,0,6,29.30,38293
0,spec lookup,What is the regulated feedback voltage (VFB) and its guaranteed tolerance?,1.00,1.00,1.00,0.60,1.00,supported,0,3,25.20,20093
2,variant,"What output voltage does the LTC3310S-1 regulate to, and how accurately?",1.00,1.00,1.00,0.70,1.00,supported,0,6,39.40,52442
1,spec lookup,What are the VIN undervoltage lockout threshold limits and the hysteresis?,1.00,0.67,1.00,0.52,1.00,supported,1,4,40.80,39455
3,ordering,What is the part marking for LTC3310HV-1#PBF and what is its temperature range?,1.00,1.00,1.00,0.67,1.00,supported,0,3,28.00,23312
4,absolute max,"What is the absolute maximum rating for VIN, and what is the maximum reflow temperature?",1.00,1.00,1.00,0.50,1.00,supported,0,3,17.80,16539
5,spec lookup,What is the guaranteed top switch current limit?,1.00,1.00,1.00,0.67,1.00,supported,0,3,24.00,20301
6,spec lookup,What switching frequency does the part run at if RT is tied to VIN?,1.00,1.00,1.00,0.81,1.00,supported,0,7,36.30,39180
8,design calculation,"For the LTC3310S at 1 MHz switching frequency, 5 V input and 1.2 V output: what RT resistor should be used and what minimum inductor value does the datasheet guideline give?",1.00,0.89,1.00,0.48,0.50,supported,0,6,39.40,56143
9,graph reading,"From the typical performance curves, approximately what is the PMOS switch on-resistance at 5 V input?",1.00,0.67,1.00,0.62,1.00,supported,1,4,42.70,37589


### Failures to look at

Rows scoring below the threshold on correctness or groundedness. Each row includes the judge's reasoning and the Phoenix trace id, so you can open the full agent run (every search, tool result, and LLM step) at http://localhost:6006 in project `datasheet-agent`.

In [10]:
THRESHOLD = 0.7
fails = results[(results["Correctness"] < THRESHOLD) | (results["Groundedness"] < THRESHOLD)]
if fails.empty:
    print(f"No question below {THRESHOLD} on correctness or groundedness.")
for _, r in fails.iterrows():
    print("=" * 110)
    print(f"[{r['app_version']}] {r['category']}: {r['query']}")
    print(f"Correctness={r['Correctness']:.2f}  Groundedness={r['Groundedness']:.2f}  verdict={r['verdict']}  "
          f"Phoenix trace={r['phoenix_trace_id']}")
    print("Expected:", GOLDEN_BY_QUERY[r["query"]]["expected_response"])
    print("Answer:  ", r["output"][:1200])

[gpt-5.5-verifier] spec lookup: What are the VIN undervoltage lockout threshold limits and the hysteresis?
Correctness=1.00  Groundedness=0.67  verdict=supported  Phoenix trace=96ca6aaeaabb65a85e09a496694b5107
Expected: VIN UVLO (VIN rising) is 2.0 V min, 2.1 V typical, 2.2 V max over temperature; UVLO hysteresis is 150 mV typical.
Answer:   For the LTC3310S / LTC3310S-1 / LTC3310 / LTC3310-1 devices covered by the datasheet, the **VIN undervoltage lockout threshold is specified for VIN rising** as:

- **Minimum:** 2.0 V
- **Typical:** 2.1 V
- **Maximum:** 2.2 V
- **Hysteresis:** 150 mV typical [p.3]

Using the typical rising threshold and typical hysteresis, the implied typical falling UVLO threshold is:

**2.1 V − 0.150 V = 1.95 V** [p.3]

The datasheet provides guaranteed min/max limits for the rising UVLO threshold; hysteresis is given as typical only.
[gpt-5.5-verifier] graph reading: From the typical performance curves, approximately what is the PMOS switch on-resistance at 5 V i

## 9. Save results

In [11]:
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
out = EVAL_DIR / f"results_{stamp}.csv"
results.to_csv(out, index=False)
leaderboard.to_csv(EVAL_DIR / f"leaderboard_{stamp}.csv")
print("saved", out)

saved evals\results_20261113-144957.csv


## 10. TruLens dashboard (optional)

The dashboard is a Streamlit app with a leaderboard, per-record drill-down, and each feedback's reasoning. It runs as a local server; stop it with `stop_dashboard(session)`.

In [12]:
RUN_DASHBOARD = False
if RUN_DASHBOARD:
    from trulens.dashboard import run_dashboard, stop_dashboard
    run_dashboard(session, port=8501)     # http://localhost:8501